# Experiment 5: Oracle ceiling of a granule (superpixel) design

Could a granule-level version of the pipeline have any headroom over the seed-only control? This notebook answers it with **no model and no fitting**: it gives the design the ground truth for free and asks how well the best possible assignment of granules could do.

**Method**
1. Cut the crop into granules: SLIC superpixels of a chosen average side, and a grid control of square blocks of the same size that ignores the image.
2. Oracle whole-image ceiling: every granule takes the ground-truth label of its majority pixels; Dice against the ground truth.
3. Oracle band ceiling: the definite seed pixels stay fixed, only the unknown band is assigned, granule by granule, by the ground-truth majority inside the band.
4. Seed-only control: the same band filled by nearest seed.

In [1]:
!pip install -q opencv-python scikit-image scipy tqdm pandas
import os, json, warnings
import numpy as np, pandas as pd, cv2
from tqdm import tqdm
from scipy.ndimage import distance_transform_edt
from skimage.segmentation import slic
warnings.filterwarnings('ignore')
try:
    from google.colab import drive; drive.mount('/content/drive', force_remount=False)
except Exception: pass

DRIVE_ROOT = "/content/drive/MyDrive/cbis_ddsm_grc_project_v3_dicom"
CONFIG = json.load(open(os.path.join(DRIVE_ROOT, "config.json")))
PRE = CONFIG.get("PREPROCESSED_DIR", os.path.join(DRIVE_ROOT, "preprocessed")); RES = CONFIG.get("RESULTS_DIR", os.path.join(DRIVE_ROOT, "results"))
ERODE, DILATE = int(CONFIG.get("TRIMAP_ERODE_ITERS", 6)), int(CONFIG.get("TRIMAP_DILATE_ITERS", 10))
SIDES = [4, 6, 8, 12, 16]            # average granule side in pixels
COMPACTNESS = 0.1                    # SLIC compactness on a 0..1 image
N_BOOT = 2000; SEED = int(CONFIG.get("SEED", 42))
OUT_DIR = os.path.join(RES, "nb9"); os.makedirs(OUT_DIR, exist_ok=True)
print("erode", ERODE, "dilate", DILATE, "| outputs ->", OUT_DIR)


Mounted at /content/drive
erode 6 dilate 10 | outputs -> /content/drive/MyDrive/cbis_ddsm_grc_project_v3_dicom/results/nb9


## Load crops, masks, split

In [ ]:
df = pd.read_csv(os.path.join(RES, "final_df_loaded.csv"))
raw_arch = np.load(os.path.join(PRE, "raw_crops.npz")); gt_arch = np.load(os.path.join(PRE, "gt_masks.npz"))
patient_of = dict(zip(df['image_id'], df['patient_id']))
sp = json.load(open(os.path.join(RES, "train_test_split.json"))); test_ids = set(sp['test_ids'])
ids = sorted(set(raw_arch.files) & set(gt_arch.files)); print(len(ids), "lesions;", len(set(ids) & test_ids), "in the test split")

1067 lesions; 269 in the test split


## Granules and ceilings

In [ ]:
K3 = np.ones((3, 3), np.uint8)
def eroded_capped(m, n):
    out = m.copy()
    for _ in range(n):
        nxt = cv2.erode(out, K3)
        if nxt.sum() == 0: break
        out = nxt
    return out
def seeds(gt):
    m = (gt > 0.5).astype(np.uint8)
    fg = eroded_capped(m, ERODE) == 1
    bg = cv2.dilate(m, K3, iterations=DILATE) == 0
    return m.astype(bool), fg, bg, ~(fg | bg)
def dice(a, b):
    s = a.sum() + b.sum(); return 1.0 if s == 0 else 2.0 * (a & b).sum() / s
def prep(raw):
    lo, hi = np.percentile(raw, [1, 99]); return np.clip((raw - lo) / (hi - lo + 1e-9), 0, 1).astype(np.float64)
def granules(img, side, kind):
    H, W = img.shape
    if kind == 'grid':
        r, c = np.arange(H) // side, np.arange(W) // side
        return (r[:, None] * (W // side + 2) + c[None, :]).astype(np.int64)
    n = max(4, int(round(H * W / float(side * side))))
    return slic(img, n_segments=n, compactness=COMPACTNESS, channel_axis=None, start_label=0, enforce_connectivity=True)
def majority(lab, gt_bool, where):
    L = lab.max() + 1
    tot = np.bincount(lab[where], minlength=L); pos = np.bincount(lab[where & gt_bool], minlength=L)
    return (pos * 2 > tot)[lab]
def one(iid):
    gt = gt_arch[iid]; img = prep(raw_arch[iid].astype(np.float32))
    m, fg, bg, unk = seeds(gt)
    if m.sum() < 20 or unk.sum() == 0: return None
    d_fg, d_bg = distance_transform_edt(~fg), distance_transform_edt(~bg)
    seed_only = fg | (unk & (d_fg < d_bg))
    row = {'image_id': iid, 'patient_id': patient_of.get(iid, iid), 'test': iid in test_ids, 'seed_only': dice(seed_only, m),
           'seed_only_band_only_share': unk.sum() / m.size}
    for kind in ('slic', 'grid'):
        for s in SIDES:
            lab = granules(img, s, kind)
            whole = majority(lab, m, np.ones_like(m))
            band = fg | (unk & majority(lab, m, unk))
            row[f'{kind}{s}_whole'] = dice(whole, m); row[f'{kind}{s}_band'] = dice(band, m)
    return row

## Run

In [ ]:
rows = [r for r in (one(i) for i in tqdm(ids, desc="lesions")) if r is not None]
R = pd.DataFrame(rows); R.to_csv(os.path.join(OUT_DIR, "granule_ceiling_per_lesion.csv"), index=False)
print(len(R), "lesions scored")

lesions: 100%|██████████| 1067/1067 [01:56<00:00,  9.17it/s]

1067 lesions scored


## Summary

In [ ]:
rng = np.random.default_rng(SEED)
def boot(d, col):
    g = d.groupby('patient_id')[col].agg(['sum', 'count']); S, C = g['sum'].values, g['count'].values
    ix = rng.integers(0, len(g), (N_BOOT, len(g))); est = S[ix].sum(1) / C[ix].sum(1)
    return S.sum() / C.sum(), *np.percentile(est, [2.5, 97.5])
def fmt(t): return f"{t[0]:.3f} [{t[1]:.3f}, {t[2]:.3f}]"
out = []
for subset, d in (("all lesions", R), ("test split", R[R.test])):
    d = d.copy()
    for s in SIDES:
        d[f'dS_{s}'] = d[f'slic{s}_band'] - d['seed_only']; d[f'dG_{s}'] = d[f'slic{s}_band'] - d[f'grid{s}_band']
        out.append({'subset': subset, 'n': len(d), 'granule side px': s, 'seed-only control': fmt(boot(d, 'seed_only')),
                    'SLIC band ceiling': fmt(boot(d, f'slic{s}_band')), 'grid band ceiling': fmt(boot(d, f'grid{s}_band')),
                    'SLIC - seed-only': fmt(boot(d, f'dS_{s}')), 'SLIC - grid': fmt(boot(d, f'dG_{s}')),
                    'SLIC whole-image ceiling': fmt(boot(d, f'slic{s}_whole')), 'grid whole-image ceiling': fmt(boot(d, f'grid{s}_whole'))})
T = pd.DataFrame(out); T.to_csv(os.path.join(OUT_DIR, "table_granule_ceiling.csv"), index=False)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 20); pd.set_option('display.max_colwidth', 40)
print(T.to_string(index=False))

a = R.copy(); verdict = []
for s in SIDES:
    dS, dG = boot(a.assign(x=a[f'slic{s}_band'] - a['seed_only']), 'x'), boot(a.assign(x=a[f'slic{s}_band'] - a[f'grid{s}_band']), 'x')
    ok = dS[0] >= 0.05 and dS[1] > 0 and dG[0] >= 0.02 and dG[1] > 0
    verdict.append(ok); print(f"side {s:2d}px: SLIC-seed_only {dS[0]:+.3f} (needs >= +0.05), SLIC-grid {dG[0]:+.3f} (needs >= +0.02) -> {'HEADROOM' if ok else 'no headroom'}")
print("\nDECLARED RULE:", "the granule design has headroom at some size" if any(verdict) else "RULED OUT: no granule size clears the rule")

     subset    n  granule side px    seed-only control    SLIC band ceiling    grid band ceiling        SLIC - seed-only             SLIC - grid SLIC whole-image ceiling grid whole-image ceiling
all lesions 1067                4 0.759 [0.744, 0.773] 0.866 [0.859, 0.873] 0.880 [0.875, 0.886]    0.107 [0.100, 0.116] -0.014 [-0.019, -0.011]     0.866 [0.857, 0.874]     0.880 [0.874, 0.886]
all lesions 1067                6 0.759 [0.744, 0.773] 0.784 [0.770, 0.797] 0.821 [0.810, 0.831]    0.025 [0.020, 0.030] -0.037 [-0.044, -0.031]     0.772 [0.755, 0.788]     0.812 [0.798, 0.825]
all lesions 1067                8 0.759 [0.744, 0.773] 0.725 [0.708, 0.741] 0.789 [0.779, 0.799] -0.034 [-0.041, -0.026] -0.064 [-0.074, -0.054]     0.707 [0.687, 0.727]     0.780 [0.767, 0.793]
all lesions 1067               12 0.759 [0.745, 0.772] 0.553 [0.529, 0.575] 0.570 [0.548, 0.591] -0.206 [-0.220, -0.192] -0.017 [-0.027, -0.007]     0.513 [0.488, 0.539]     0.539 [0.513, 0.565]
all lesions 1067         

In [ ]:
checks = {
  "lesions scored": len(R) > 0,
  "metrics in [0,1]": bool(R.drop(columns=['image_id', 'patient_id', 'test']).stack().between(0, 1).all()),
  "oracle whole-image >= oracle band-only is NOT assumed; both present": all(f'slic{s}_whole' in R and f'slic{s}_band' in R for s in SIDES),
  "outputs saved": os.path.exists(os.path.join(OUT_DIR, "table_granule_ceiling.csv")),
}
for k, v in checks.items(): print("  OK  " if v else "  FAIL", k)
assert all(checks.values())

  OK   lesions scored
  OK   metrics in [0,1]
  OK   oracle whole-image >= oracle band-only is NOT assumed; both present
  OK   outputs saved
